In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

try:
    df = pd.read_csv('drug.csv')
    print("Successfully loaded 'drug.csv'")
except FileNotFoundError:
    print("Warning: 'drug.csv' not found. Creating a sample dataset.")
    data = {
        'Age': np.random.randint(15, 75, 200),
        'Sex': np.random.choice(['F', 'M'], 200),
        'BP': np.random.choice(['LOW', 'NORMAL', 'HIGH'], 200),
        'Cholesterol': np.random.choice(['NORMAL', 'HIGH'], 200),
        'Na_to_K': np.random.uniform(5, 35, 200),
        'Drug': np.random.choice(['drugA', 'drugB', 'drugC', 'drugX', 'drugY'], 200)
    }
    df = pd.DataFrame(data)

    df.loc[df['Na_to_K'] > 15, 'Drug'] = 'drugY'
    df.loc[(df['BP'] == 'HIGH') & (df['Age'] > 50), 'Drug'] = 'drugB'
    df.loc[(df['BP'] == 'LOW') & (df['Cholesterol'] == 'HIGH'), 'Drug'] = 'drugC'
    df.loc[(df['Age'] < 30) & (df['BP'] == 'NORMAL'), 'Drug'] = 'drugX'
    df.loc[(df['Sex'] == 'F') & (df['BP'] == 'HIGH'), 'Drug'] = 'drugA'

print("\nDataset head:")
print(df.head())
print("\nDataset info:")
df.info()

if 'Drug' not in df.columns:
    print("\nError: Target column 'Drug' not found. Aborting.")
else:
    X = df.drop('Drug', axis=1)
    y = df['Drug']

    categorical_cols = X.select_dtypes(include=['object', 'category']).columns

    if len(categorical_cols) > 0:
        print(f"\nConverting categorical columns to numbers: {list(categorical_cols)}")
        X_processed = pd.get_dummies(X, columns=categorical_cols, drop_first=False)
        print("Processed features head:")
        print(X_processed.head())
    else:
        print("\nNo categorical columns found. Assuming all features are numeric.")
        X_processed = X


    X_train, X_test, y_train, y_test = train_test_split(
        X_processed, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"\nData split: {len(X_train)} training samples, {len(X_test)} testing samples.")

    model = DecisionTreeClassifier(criterion='entropy', random_state=42)

    model.fit(X_train, y_train)
    print("Successfully trained Decision Tree model.")

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    print("\n--- Final Results ---")
    print(f"The accuracy of the Decision Tree algorithm is: {accuracy:.4f}")

Successfully loaded 'drug.csv'

Dataset head:
   Age Sex      BP Cholesterol  Na_to_K   Drug
0   23   F    HIGH        HIGH   25.355  drugY
1   47   M     LOW        HIGH   13.093  drugC
2   47   M     LOW        HIGH   10.114  drugC
3   28   F  NORMAL        HIGH    7.798  drugX
4   61   F     LOW        HIGH   18.043  drugY

Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Age          200 non-null    int64  
 1   Sex          200 non-null    object 
 2   BP           200 non-null    object 
 3   Cholesterol  200 non-null    object 
 4   Na_to_K      200 non-null    float64
 5   Drug         200 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 9.5+ KB

Converting categorical columns to numbers: ['Sex', 'BP', 'Cholesterol']
Processed features head:
   Age  Na_to_K  Sex_F  Sex_M  BP_HIGH  BP_LOW  BP_NORMAL